<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week4_neural_networks/Week4_Lesson7_Lecture.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 IB CS — Неделя 4 · Урок 7 (Лекция)
## Нейронные сети — ANN, CNN, Backpropagation
### A4.3.8 (ANN) + A4.3.9 (CNN) + training (backprop, gradient descent)

> ⏱ Длительность: 2 академических часа.
> 📘 Источники: Baumgarten (Hodder IBDP) + MacKenty & Stephenson (Oxford 2025).

---

### 🎯 Что покрываем сегодня (по syllabus):

| Statement | Тема | Command term |
|---|---|---|
| **A4.3.8** | Structure & function of ANNs | *Outline* |
| **A4.3.9** | CNNs for spatial hierarchies in images | *Describe* |
| **+** | Backpropagation, gradient descent (training process) | (часть A4.3.8) |
| **+** | Activation functions: ReLU, Sigmoid, Softmax, tanh | |

---

### ⚠️ Важно перед стартом

> Это **самая сложная** тема в ML-разделе IB. **Не пытайтесь** понять backpropagation математически — **синтаксис IB**:
>
> 1. **Уметь нарисовать** структуру ANN
> 2. **Описать словами** что делает каждый слой
> 3. **Объяснить** в общих чертах, как сеть учится
>
> Из учебника Baumgarten (прямая цитата): *"The training process is more complex and is far beyond the scope of your course."*
>
> 💎 **СЕКРЕТ #1:** на IB **не нужны** формулы chain rule и derivatives. Нужна **концепция**.


## 🧠 Часть 1 · A4.3.8 Структура ANN

> **Definition (выучить!):** *An artificial neural network (ANN) is a computational model designed to mimic the human brain's interconnected neuron structure to process information. It consists of layers of nodes (called **perceptrons**) connected by pathways that transmit signals.*

### 🧩 Три типа слоёв

| Слой | Что делает | Сколько нейронов? |
|---|---|---|
| **Input layer** | принимает фичи (1 нейрон на 1 фичу) | = число фич |
| **Hidden layer(s)** | извлекает паттерны (могут быть несколько) | произвольно (hyperparameter) |
| **Output layer** | даёт финальное предсказание | = число классов (classif.) или 1 (regression) |

> 🔑 **Fully connected network** = каждый нейрон одного слоя связан с КАЖДЫМ нейроном следующего. Это **стандарт**.


In [ ]:
# Визуализация структуры ANN: 3 inputs → 4 hidden → 2 outputs
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(12, 7))

# Координаты нейронов
layer_sizes = [3, 4, 2]
layer_x = [0.1, 0.5, 0.9]
colors = ['#FFD93D', '#4ECDC4', '#FF6B6B']
labels = ['Input layer\n(I1, I2, I3)', 'Hidden layer\n(H1, H2, H3, H4)', 'Output layer\n(O1, O2)']

positions = {}
for i, (size, x, color) in enumerate(zip(layer_sizes, layer_x, colors)):
    y_positions = np.linspace(0.2, 0.8, size)
    for j, y in enumerate(y_positions):
        circle = patches.Circle((x, y), 0.04, color=color, ec='black', lw=2, zorder=3)
        ax.add_patch(circle)
        positions[(i, j)] = (x, y)
        # Подписи нейронов
        if i == 0:
            ax.text(x - 0.07, y, f'I{j+1}', fontsize=10, ha='right', va='center')
        elif i == 1:
            ax.text(x, y + 0.06, f'H{j+1}', fontsize=9, ha='center')
        else:
            ax.text(x + 0.07, y, f'O{j+1}', fontsize=10, ha='left', va='center')

# Связи между слоями
for i in range(len(layer_sizes) - 1):
    for j in range(layer_sizes[i]):
        for k in range(layer_sizes[i+1]):
            x1, y1 = positions[(i, j)]
            x2, y2 = positions[(i+1, k)]
            ax.plot([x1, x2], [y1, y2], color='gray', alpha=0.4, lw=0.7, zorder=1)

# Подписи слоёв
for x, label in zip(layer_x, labels):
    ax.text(x, 0.05, label, ha='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Полностью связанная ANN: 3 → 4 → 2 (fully connected)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

print("📊 Подсчёт параметров:")
print(f"  Веса между Input(3) и Hidden(4): 3 × 4 = 12")
print(f"  Веса между Hidden(4) и Output(2): 4 × 2 = 8")
print(f"  Bias в Hidden: 4")
print(f"  Bias в Output: 2")
print(f"  ИТОГО параметров: 12 + 8 + 4 + 2 = 26")

### 🔬 Один перцептрон под микроскопом

> **Perceptron** (по Baumgarten): *the data structure at the heart of an artificial neural network; it represents a single artificial neuron that takes in multiple inputs and weights, and generates an output value.*

**Что происходит внутри одного нейрона (5 шагов):**

| Шаг | Операция |
|---|---|
| 1. **Inputs** | получает значения от нейронов предыдущего слоя |
| 2. **Weights** | каждый input умножается на свой weight (важность) |
| 3. **Summation** | все произведения складываются |
| 4. **Bias** | прибавляется bias (смещение) |
| 5. **Activation** | результат проходит через activation function |

**Формула одного нейрона:**

$$y = \text{ReLU}(x_1 w_1 + x_2 w_2 + x_3 w_3 + b)$$

где: $x_i$ = inputs, $w_i$ = weights, $b$ = bias, ReLU = activation function.


In [ ]:
# Worked example из Baumgarten (стр. 47): расчёт одного перцептрона
# Inputs: 1.3, 4.2, 0.0, 2.7
# Weights: -3.1, 1.6, 2.9, 2.7
# Bias: -5.2
# Activation: ReLU

inputs = np.array([1.3, 4.2, 0.0, 2.7])
weights = np.array([-3.1, 1.6, 2.9, 2.7])
bias = -5.2

# Шаг 1-3: умножение и сумма
summation = np.dot(inputs, weights)
print(f"Шаг 1-3 — Summation:")
print(f"  (1.3 × -3.1) + (4.2 × 1.6) + (0.0 × 2.9) + (2.7 × 2.7)")
print(f"  = {1.3*-3.1:.2f} + {4.2*1.6:.2f} + {0.0*2.9:.2f} + {2.7*2.7:.2f}")
print(f"  = {summation:.2f}")

# Шаг 4: добавление bias
z = summation + bias
print(f"\nШаг 4 — Add bias: {summation:.2f} + ({bias}) = {z:.2f}")

# Шаг 5: ReLU activation
output = max(0, z)  # ReLU: max(0, x)
print(f"\nШаг 5 — ReLU({z:.2f}) = {output:.2f}")
print(f"\n💎 Этот нейрон ВЫХОД = {output:.2f}")

> 💎 **СЕКРЕТ #2 (часто на IB!):** *"Describe one step of forward propagation"* **[3]** — образец на 3/3:
> 1. *"Each neuron in the layer receives inputs from all neurons in the previous layer."*
> 2. *"Inputs are multiplied by their respective weights, summed together, and a bias is added."*
> 3. *"The result is passed through an activation function (such as ReLU) to produce the neuron's output, which is then sent to the next layer."*


## ⚡ Часть 2 · Activation Functions

> **Definition:** *A mathematical function applied to the output of a neuron to determine whether it should be "activated" (output non-zero) and introduce non-linearity.*

### 🎯 Зачем нужна activation function?

> Без activation — вся сеть **сводится к одной линейной функции** (любое число линейных слоёв = один линейный слой).
> Activation function вносит **non-linearity**, позволяя сети моделировать **сложные паттерны**.

### 4 главных activation function (по IB):

| Функция | Формула | Output range | Где используется |
|---|---|---|---|
| **ReLU** | $\max(0, x)$ | $[0, +\infty)$ | **default** для hidden layers |
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | $(0, 1)$ | binary classification output |
| **Tanh** | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $(-1, 1)$ | hidden layers (центр в 0) |
| **Softmax** | $\frac{e^{x_i}}{\sum_j e^{x_j}}$ | $(0, 1)$, sum=1 | **multi-class** output (вероятности) |


In [ ]:
# Визуализация 4 activation functions
x = np.linspace(-5, 5, 200)
relu = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(x, relu, color='#FF6B6B', linewidth=2.5)
axes[0].set_title('ReLU — default для hidden', fontweight='bold')
axes[0].axhline(0, color='gray', linewidth=0.5); axes[0].axvline(0, color='gray', linewidth=0.5)
axes[0].grid(alpha=0.3)

axes[1].plot(x, sigmoid, color='#4ECDC4', linewidth=2.5)
axes[1].set_title('Sigmoid — binary output', fontweight='bold')
axes[1].axhline(0, color='gray', linewidth=0.5); axes[1].axhline(1, color='gray', linewidth=0.5, ls='--')
axes[1].axvline(0, color='gray', linewidth=0.5)
axes[1].grid(alpha=0.3)

axes[2].plot(x, tanh, color='#FFD93D', linewidth=2.5)
axes[2].set_title('Tanh — hidden (centred at 0)', fontweight='bold')
axes[2].axhline(0, color='gray', linewidth=0.5); axes[2].axvline(0, color='gray', linewidth=0.5)
axes[2].axhline(1, color='gray', linewidth=0.5, ls='--')
axes[2].axhline(-1, color='gray', linewidth=0.5, ls='--')
axes[2].grid(alpha=0.3)

# Softmax — для 3 классов
logits = np.array([[xi-1, xi, xi+1] for xi in x])
softmax_vals = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
axes[3].plot(x, softmax_vals[:, 0], label='Class 1', linewidth=2)
axes[3].plot(x, softmax_vals[:, 1], label='Class 2', linewidth=2)
axes[3].plot(x, softmax_vals[:, 2], label='Class 3', linewidth=2)
axes[3].set_title('Softmax — multi-class output', fontweight='bold')
axes[3].legend(fontsize=8); axes[3].grid(alpha=0.3)
axes[3].set_ylabel('Probability')

plt.suptitle('4 главные activation functions (по IB syllabus)',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()

### 🆚 ReLU vs Sigmoid — почему ReLU стал стандартом?

| Критерий | ReLU | Sigmoid |
|---|---|---|
| Скорость вычисления | **очень быстро** (просто max) | медленнее (exp) |
| Vanishing gradient | **редко** | **частая проблема** |
| Output range | [0, +∞) | (0, 1) |
| Где используется | hidden layers | output (binary) |

> 💎 **СЕКРЕТ #3:** *"Why is ReLU preferred over Sigmoid in deep networks?"* **[3]** — на 3/3:
> 1. **Vanishing gradient problem**: Sigmoid имеет очень малые градиенты при больших |x| → backprop "затухает" в глубоких сетях
> 2. **Computational efficiency**: ReLU = просто `max(0, x)`, Sigmoid требует exp() — медленнее
> 3. **Biological plausibility**: ReLU имитирует "all-or-nothing" поведение биологических нейронов


## 🔄 Часть 3 · Forward & Backpropagation (training)

### Цикл обучения нейросети — 4 шага (по MacKenty/Stephenson):

| Шаг | Что происходит |
|---|---|
| 1. **Forward propagation** | данные идут от input → output, считается prediction |
| 2. **Calculate loss** | сравниваем prediction с истинным значением через loss function |
| 3. **Backpropagation** | ошибка распространяется обратно, считаются градиенты для каждого веса |
| 4. **Update weights** | веса обновляются через gradient descent с learning rate |

Повторяем для **N эпох** (epoch = один полный проход через весь training set).


In [ ]:
# Визуализация цикла обучения
fig, ax = plt.subplots(figsize=(13, 6))

# 4 этапа как круг
import matplotlib.patches as mpatches
stages = [
    ('1. Forward\nPropagation', 0.2, 0.7, '#4ECDC4',
     'Input → Hidden → Output\ny_pred = f(W·x + b)'),
    ('2. Calculate\nLoss', 0.55, 0.85, '#FFD93D',
     'Loss = MSE / Cross-entropy\nL = (y_pred - y_true)²'),
    ('3. Back-\npropagation', 0.85, 0.7, '#FF6B6B',
     'Compute ∂L/∂w for each weight\n(chain rule)'),
    ('4. Update\nWeights', 0.55, 0.25, '#A78BFA',
     'w_new = w_old - lr · ∂L/∂w\n(gradient descent)'),
]

for label, x, y, color, formula in stages:
    box = mpatches.FancyBboxPatch((x-0.08, y-0.06), 0.16, 0.12,
                                    boxstyle="round,pad=0.02",
                                    facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(x, y - 0.13, formula, ha='center', va='center', fontsize=8, style='italic')

# Стрелки между этапами
arrow_props = dict(arrowstyle='->', lw=2, color='black')
ax.annotate('', xy=(0.47, 0.82), xytext=(0.30, 0.74), arrowprops=arrow_props)
ax.annotate('', xy=(0.77, 0.74), xytext=(0.63, 0.82), arrowprops=arrow_props)
ax.annotate('', xy=(0.63, 0.30), xytext=(0.79, 0.62), arrowprops=arrow_props)
ax.annotate('', xy=(0.28, 0.62), xytext=(0.47, 0.30), arrowprops=arrow_props)

ax.text(0.5, 0.5, 'EPOCH\n(полный цикл)',
        ha='center', va='center', fontsize=14, fontweight='bold',
        bbox=dict(boxstyle="round", facecolor='lightgray', edgecolor='black'))

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Цикл обучения нейросети (1 epoch)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 🎯 Backpropagation — суть в одном абзаце

> **Definition (Baumgarten):** *Backpropagation is the most commonly used technique for training ANNs. The gradient of the loss function is calculated and used to update parameters such as weights, in the opposite direction of the gradient to reduce the overall error.*

**Простыми словами:**
1. Сеть делает предсказание (forward pass).
2. Считаем, насколько прогноз отличается от правильного ответа (**loss**).
3. **Распространяем ошибку обратно** через слои — кто из весов "виноват" больше всех?
4. Корректируем веса **в обратном направлении** градиента (gradient descent).

> 💎 **СЕКРЕТ #4 (часто на IB):** *"Define backpropagation and outline its role in learning"* **[3]** — Baumgarten Review #18:
> 1. **Definition** (1 балл): "Backpropagation is an algorithm that propagates the prediction error backwards from the output to the input layer."
> 2. **Calculates gradients** (1 балл): "It computes how much each weight contributed to the error using the gradient of the loss function."
> 3. **Updates weights** (1 балл): "These gradients are used to update weights via gradient descent, reducing the loss over multiple epochs."

---

### 📉 Gradient Descent + Learning Rate

> **Gradient descent** — оптимизационный алгоритм, который "спускается" вниз по поверхности loss function.
>
> **Learning rate (η)** — насколько большие шаги делаем.

**Trade-off learning rate:**
- **Слишком большой** (η = 1.0) → "перепрыгиваем" минимум, loss скачет
- **Слишком малый** (η = 0.0001) → обучение **очень медленное**
- **Оптимальный** (η ≈ 0.001–0.01) → стабильное и быстрое обучение


In [ ]:
# Визуализация gradient descent — как learning rate влияет на convergence
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Параболическая loss function
def loss(w):
    return (w - 3)**2 + 1

def gradient(w):
    return 2*(w - 3)

w_range = np.linspace(-2, 8, 100)
losses = [loss(w) for w in w_range]

learning_rates = [0.05, 0.5, 1.1]
titles = ['Small LR (η=0.05)\n— медленно, но стабильно',
          'Optimal LR (η=0.5)\n— быстро и стабильно',
          'Large LR (η=1.1)\n— расходится!']
colors = ['steelblue', 'green', 'red']

for ax, lr, title, color in zip(axes, learning_rates, titles, colors):
    ax.plot(w_range, losses, 'k-', linewidth=2, alpha=0.5)

    # Симулируем gradient descent
    w = -1.5  # стартовая точка
    path_w = [w]
    path_l = [loss(w)]
    for _ in range(15):
        w = w - lr * gradient(w)
        path_w.append(w)
        path_l.append(loss(w))
        if abs(w) > 10:  # для расходящегося случая
            break

    ax.plot(path_w, path_l, 'o-', color=color, markersize=8, linewidth=1.5,
            label=f'{len(path_w)-1} шагов')
    ax.scatter([path_w[0]], [path_l[0]], s=200, color=color, marker='*',
               edgecolor='black', linewidth=2, zorder=5, label='start')
    ax.scatter([3], [1], s=200, color='gold', marker='X',
               edgecolor='black', linewidth=2, zorder=5, label='minimum')

    ax.set_xlabel('weight w'); ax.set_ylabel('Loss')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.set_ylim(-2, 50)

plt.tight_layout(); plt.show()

### ⚠️ Overfitting в нейросетях

> **Common mistake (Baumgarten):** *"Overcomplicating the model architecture can lead to overfitting and high computational costs. Start with a simple architecture and gradually increase complexity, if necessary."*

**Признаки overfitting:**
- Training accuracy >> Test accuracy (большой разрыв)
- Loss на training падает, loss на validation **растёт**
- Сеть имеет **много параметров** на **мало данных**

**Методы борьбы:**
1. **Dropout** — случайно "выключать" нейроны во время обучения
2. **L2 regularization** — штраф за большие веса
3. **Early stopping** — остановиться, когда validation loss перестал падать
4. **Data augmentation** — увеличить датасет искусственно (вращение, шум)
5. **Уменьшить архитектуру** — меньше слоёв / нейронов


## 🖼️ Часть 4 · A4.3.9 Convolutional Neural Networks (CNN)

> **Definition (Baumgarten):** *A CNN extends ANN architecture by using additional layers of calculations prior to processing data through a fully connected ANN. It applies **convolution** — a mathematical operation that uses filtering functions to compute distinctive features from images.*

### 🎯 Зачем CNN, если есть ANN?

> Изображение 100×100 пикселей × 3 канала (RGB) = **30 000 значений**.
> Если каждое подать в ANN → нужны **миллионы параметров** → overfitting + медленно.
>
> **CNN** сначала **сжимает** изображение через convolution + pooling, **извлекая важные паттерны**.

### 🏗️ Архитектура CNN (5 типов слоёв)

| Слой | Что делает | Зачем |
|---|---|---|
| **Input** | принимает raw pixels | height × width × channels |
| **Convolutional** | применяет фильтры (kernels), создаёт feature maps | detect edges, textures, patterns |
| **Activation** (ReLU) | вносит non-linearity | важно после каждой свёртки |
| **Pooling** | уменьшает размер feature maps | reduce computation + overfitting |
| **Fully connected** | классическая ANN в конце | финальная классификация |
| **Output** (softmax) | вероятности классов | sum to 1 |


In [ ]:
# Визуализация архитектуры CNN
fig, ax = plt.subplots(figsize=(15, 5))

stages = [
    ('Input\n28×28×1', 0.05, '#FFD93D'),
    ('Conv\n3×3 kernels', 0.20, '#4ECDC4'),
    ('ReLU', 0.32, '#FF6B6B'),
    ('Max Pool\n2×2', 0.42, '#A78BFA'),
    ('Conv\n3×3 kernels', 0.55, '#4ECDC4'),
    ('ReLU', 0.65, '#FF6B6B'),
    ('Max Pool', 0.74, '#A78BFA'),
    ('Flatten', 0.83, '#FFD93D'),
    ('Fully\nConnected', 0.91, '#FFA07A'),
    ('Softmax\n→ 10 classes', 0.99, '#90EE90'),
]

for label, x, color in stages:
    box = mpatches.FancyBboxPatch((x-0.04, 0.4), 0.07, 0.25,
                                    boxstyle="round,pad=0.01",
                                    facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x, 0.525, label, ha='center', va='center', fontsize=9, fontweight='bold')

# Стрелки
for i in range(len(stages) - 1):
    x1 = stages[i][1] + 0.025
    x2 = stages[i+1][1] - 0.045
    ax.annotate('', xy=(x2, 0.525), xytext=(x1, 0.525),
                arrowprops=dict(arrowstyle='->', lw=1.5))

ax.text(0.05, 0.85, '📷 Изображение', ha='center', fontsize=10, style='italic')
ax.text(0.30, 0.85, '🔍 Feature extraction', ha='center', fontsize=10, style='italic')
ax.text(0.65, 0.85, '🔍 Higher features', ha='center', fontsize=10, style='italic')
ax.text(0.91, 0.85, '🎯 Classification', ha='center', fontsize=10, style='italic')

ax.text(0.30, 0.20, 'Layer 1: Edges, simple shapes',
        ha='center', fontsize=9, color='gray')
ax.text(0.65, 0.20, 'Layer 2: Complex patterns (eyes, fur)',
        ha='center', fontsize=9, color='gray')

ax.set_xlim(0, 1.05); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Типичная архитектура CNN для image classification',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

### 🔍 Convolutional Layer — самое интересное

> **Kernel (filter)** — маленькая матрица (часто 3×3 или 5×5), которая "скользит" по изображению и **извлекает паттерны**.

**Примеры известных фильтров (Baumgarten, p. 58):**

| Фильтр | Назначение |
|---|---|
| Vertical edge detector | находит вертикальные края |
| Horizontal edge detector | находит горизонтальные края |
| Sharpening | усиливает контраст |
| Gaussian blur | размывает (убирает шум) |

> 💎 **СЕКРЕТ #5:** *"Strength of CNNs lies in their ability to LEARN the most appropriate filters for a given task through backpropagation"* (Baumgarten).
> CNN **не используют готовые фильтры** — она **сама учит** их во время training.

**Как работает свёртка (стрид):**
- Фильтр размером 3×3 "скользит" по изображению
- На каждой позиции считает **скалярное произведение** (dot product)
- Результат → один пиксель в **feature map**


In [ ]:
# Демо: применяем vertical edge detector к простой картинке
from scipy.signal import convolve2d

# Простая картинка: левая половина светлая, правая тёмная
image = np.zeros((20, 20))
image[:, :10] = 1.0  # белая половина
# Добавим шум
image += np.random.normal(0, 0.05, image.shape)

# Vertical edge detection kernel (Sobel)
kernel = np.array([
    [1, 0, -1],
    [2, 0, -2],
    [1, 0, -1]
])

feature_map = convolve2d(image, kernel, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image\n(20×20)', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(kernel, cmap='RdBu_r', vmin=-2, vmax=2)
axes[1].set_title('Vertical Edge Kernel\n(3×3)', fontweight='bold')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{kernel[i, j]:+d}',
                    ha='center', va='center', fontsize=14, fontweight='bold')
axes[1].set_xticks([]); axes[1].set_yticks([])

axes[2].imshow(np.abs(feature_map), cmap='hot')
axes[2].set_title('Feature Map после Convolution\n(вертикальный край виден!)',
                  fontweight='bold')
axes[2].axis('off')

plt.tight_layout(); plt.show()
print("💎 Видите, как kernel выделил ВЕРТИКАЛЬНУЮ границу между светлой и тёмной зоной?")

### 🌊 Pooling Layer

> **Зачем нужен?** (3 причины из Baumgarten):
> 1. **Reduce dimensions** → меньше параметров → быстрее
> 2. **Reduce overfitting** → меньше внимания к мелким деталям
> 3. **Translation invariance** → объект распознаётся в разных позициях

**Два типа pooling:**
- **Max pooling** — берёт **максимум** из окна (например, 2×2) → сохраняет **самые яркие** признаки
- **Average pooling** — берёт **среднее** → smoother, меньше отбрасывает информацию

> 💎 **Max pooling** — самый распространённый. На IB *"Outline ONE method of pooling"* — обычно отвечаем про max pooling.


In [ ]:
# Демо: max pooling
demo_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [3, 1, 7, 8],
    [2, 4, 5, 6],
])

# Max pooling 2×2
pooled = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        pooled[i, j] = demo_map[2*i:2*i+2, 2*j:2*j+2].max()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(demo_map, cmap='Blues')
axes[0].set_title('Feature Map (4×4)', fontweight='bold')
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{demo_map[i,j]}', ha='center', va='center',
                     fontsize=16, fontweight='bold')
# Рисуем границы 2×2 регионов
for i in [0, 2]:
    for j in [0, 2]:
        rect = mpatches.Rectangle((j-0.5, i-0.5), 2, 2, linewidth=3,
                                    edgecolor='red', facecolor='none')
        axes[0].add_patch(rect)
axes[0].set_xticks([]); axes[0].set_yticks([])

axes[1].imshow(pooled, cmap='Blues')
axes[1].set_title('After Max Pooling 2×2 (→ 2×2)', fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{int(pooled[i,j])}', ha='center', va='center',
                     fontsize=20, fontweight='bold')
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.tight_layout(); plt.show()
print("💎 Max pooling берёт максимум из каждого 2×2 окна. 4×4 → 2×2.")

### 🎯 CNN vs ANN — сравнительная таблица для IB

| Критерий | ANN | CNN |
|---|---|---|
| Входные данные | плоский вектор фич | **изображение** (matrix) |
| Использует свёртку? | нет | **да** |
| Pooling? | нет | **да** |
| Сохраняет spatial structure? | **нет** (теряется при flatten) | **да** |
| Главное применение | tabular data, regression | **images**, video, audio |
| Число параметров | очень много | **меньше** (благодаря shared weights) |

> 💎 **СЕКРЕТ #6:** *"Describe how CNNs adaptively learn spatial hierarchies of features in images"* — **A4.3.9 точная формулировка syllabus**. Образец на полный балл:
>
> 1. *"Early layers detect simple features like edges and textures."*
> 2. *"Middle layers combine these into more complex shapes (eyes, ears)."*
> 3. *"Deeper layers recognize entire objects (faces of cats vs dogs)."*
> 4. *"This hierarchical learning happens automatically through backpropagation, without manual feature engineering."*


## 📝 Часть 5 · Mini Exam Practice (в классе)

### Вопрос 1 (Baumgarten Review #17, p. 39)
> *A financial institution employs an artificial neural network to predict loan default risk based on customer profiles.*
>
> **a)** *Identify* one type of layer often used in neural networks and its purpose. **[2]**
> **b)** *Describe* why overfitting might be a concern in neural networks. **[3]**
> **c)** *Describe* how a neural network can be trained to minimize prediction error in this financial context. **[3]**

> 💎 **Разбор (c) [3]:**
> 1. **Forward propagation** producing initial prediction (1 балл)
> 2. **Loss calculation** (binary cross-entropy для default/no-default) (1 балл)
> 3. **Backpropagation** + gradient descent → updates weights to minimize loss over epochs (1 балл)

---

### Вопрос 2 (Baumgarten Review #18, p. 39)
> *An energy company uses a neural network to forecast electricity demand based on weather conditions and historical usage.*
>
> **a) i)** *Identify* whether this would be regression or classification based. **[1]**
> **a) ii)** *Outline* the significance of that on its design. **[2]**
> **b)** *Outline* one type of activation function used in neural networks and its purpose. **[2]**
> **c)** *Outline* which activation function would most likely be suitable for the **output layer** in this scenario, and why. **[2]**
> **d)** *Describe* why deep neural networks might be more effective than shallow networks for this task. **[3]**
> **e)** *Define* "backpropagation" and *outline* its role in learning. **[3]**

> 💎 **Разбор (a):** Regression (electricity demand — непрерывная величина).
> 💎 **Разбор (c):** **Linear** или **ReLU** (для regression output должен быть непрерывным; **НЕ** sigmoid/softmax, потому что они ограничены).
> 💎 **Разбор (d):** Глубокие сети **иерархически** извлекают паттерны: первые слои — простые (температура→демand), глубже — комплексные (день недели + сезон + температура → точный прогноз).


## ✅ Чек-лист после Урока 7

К следующему уроку (семинар MNIST) вы должны:

### A4.3.8 ANN
- [ ] Знать **3 типа слоёв** (input, hidden, output) и их роли
- [ ] Описать **5 шагов** работы одного перцептрона
- [ ] Знать **4 activation functions** + когда какую использовать
- [ ] Уметь **рассчитать** forward pass для одного нейрона (worked example)
- [ ] Объяснить **backpropagation** в 3 идеях
- [ ] Понимать роль **learning rate** (trade-off)

### A4.3.9 CNN
- [ ] Знать **5 типов слоёв CNN** и их роли
- [ ] Объяснить, что такое **kernel/filter**
- [ ] Знать, что такое **feature map**
- [ ] Различать **max pooling** и **average pooling**
- [ ] Объяснить **spatial hierarchies** (A4.3.9 exact syllabus wording)

### Общее
- [ ] Понимать разницу **CNN vs ANN** (когда что)
- [ ] Знать концепцию **overfitting** в нейросетях + 2-3 метода борьбы

---

### 📚 ДЗ (см. `Week4_HW1_Theory.ipynb`)

1. Нарисовать схему ANN для задачи 3 inputs → 2 outputs
2. Расчёт forward pass **вручную** для одного нейрона
3. ReLU vs Sigmoid сравнение
4. IB-style вопросы по A4.3.8 и A4.3.9

---

> 🎓 **Финальный секрет Урока 7:**
> Нейросети **пугают** студентов из-за математики backpropagation.
> Но **на IB математика НЕ спрашивается** (прямая цитата Baumgarten: *"beyond the scope of your course"*).
> Что спрашивают: **структура**, **роли слоёв**, **зачем activation function**, **что такое forward/back propagation в общем**.
>
> 💎 Выучите термины — и **гарантированные баллы** ваши.
